# Notebook 05 — Event-Count-Balanced Purged Anchored Walk-Forward Design

## Purpose

Freeze the primary model-selection folds **before** Optuna and algorithm
comparison.

The input population is the Stage 04 primary long candidate set generated by
the confirmation-gated ZigZag.

## Validation design

- Train-only frozen trading calendar.
- Five anchored walk-forward folds.
- Calendar-date boundaries; no random row split.
- The first validation block starts after 50% of train-only trading dates.
- The remaining validation horizon is partitioned into five contiguous calendar
  blocks with approximately equal **candidate-event counts**.
- Candidate start-date counts are the only balancing input.
- `meta_label` is not used to construct fold boundaries.
- All candidate events sharing the same start date remain together.
- 30-trading-day conservative gap immediately before every validation block.
- Event-end overlap purging.
- No model fitting.
- No unseen-test access.


In [1]:
from __future__ import annotations

from pathlib import Path
import json
import sys
import time

import numpy as np
import pandas as pd
import yaml

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("numpy:", np.__version__)


Python: 3.12.2
pandas: 2.2.3
numpy: 1.26.4


## 1. Locate repository and import validation modules

In [2]:
def locate_repository_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "configs").exists() and (candidate / "notebooks").exists():
            return candidate
    raise FileNotFoundError(
        "Repository root was not found. Open the project root in VS Code."
    )


REPOSITORY_ROOT = locate_repository_root(Path.cwd())

if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))

from src.utils.paths import (
    data_paths,
    repository_result_paths,
    resolve_data_root,
)
from src.utils.reproducibility import (
    git_commit_sha,
    software_manifest,
    stable_object_hash,
)
from src.validation.folds import (
    FOLD_DESIGN_SCHEMA_VERSION,
    build_anchored_event_balanced_folds,
    candidate_event_counts_by_calendar_date,
    folds_to_frame,
    normalize_trading_calendar,
)
from src.validation.purged_walk_forward import (
    audit_anchored_training_sets,
    audit_fold_integrity,
    audit_validation_event_count_balance,
    audit_validation_window_overlap,
    fold_membership_masks,
    normalize_candidate_event_panel,
    summarize_fold,
)

print("Repository root:", REPOSITORY_ROOT)


Repository root: E:\Iran_stock_trade\financial-ml-decision-support


## 2. Load frozen Stage 04 and Stage 05 policies

In [3]:
def load_yaml(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as file_obj:
        value = yaml.safe_load(file_obj)
    if not isinstance(value, dict):
        raise ValueError(f"Configuration must be a mapping: {path}")
    return value


paths_config = load_yaml(REPOSITORY_ROOT / "configs" / "paths.yaml")
validation_config = load_yaml(
    REPOSITORY_ROOT / "configs" / "validation.yaml"
)

DATA_ROOT = resolve_data_root(paths_config, REPOSITORY_ROOT)
DATA_PATHS = data_paths(DATA_ROOT, paths_config)
RESULT_PATHS = repository_result_paths(
    REPOSITORY_ROOT,
    paths_config,
)

CANDIDATE_TRAIN_PATH = DATA_PATHS["candidate_data"] / "train"
LABELED_TRAIN_PATH = DATA_PATHS["labeled_data"] / "train"

stage04_policy_path = (
    RESULT_PATHS["manifests"] / "04_candidate_filter_policy.json"
)
approved_features_path = (
    RESULT_PATHS["manifests"] / "04_approved_model_features.csv"
)
frozen_universe_path = (
    RESULT_PATHS["manifests"] / "02_frozen_universe.csv"
)

for required_path in [
    stage04_policy_path,
    approved_features_path,
    frozen_universe_path,
]:
    if not required_path.exists():
        raise FileNotFoundError(
            f"Required frozen-stage artifact is missing: {required_path}"
        )

stage04_policy = json.loads(
    stage04_policy_path.read_text(encoding="utf-8")
)
primary_validation = validation_config["primary_validation"]

assert (
    FOLD_DESIGN_SCHEMA_VERSION
    == "stage05_v2_candidate_event_count_balanced_contiguous"
)
assert primary_validation["validation_window_partition"] == (
    "approximately_equal_candidate_event_count_contiguous_calendar_blocks"
)
assert primary_validation["boundary_construction_input"] == (
    "candidate_event_start_date_counts_only"
)
assert (
    primary_validation["label_balance_used_for_fold_boundaries"]
    is False
)
assert (
    primary_validation["target_column_used_for_fold_boundaries"]
    is False
)

assert validation_config["status"] == "stage_05_configured_v2"
assert stage04_policy["primary_side"] == "long"
assert np.isclose(
    float(stage04_policy["primary_threshold_fraction"]),
    0.15,
)
assert np.isclose(
    float(
        primary_validation[
            "candidate_filter_threshold_fraction"
        ]
    ),
    0.15,
)
assert primary_validation["candidate_scope"] == (
    "stage_04_primary_long_candidates"
)
assert primary_validation["unseen_test_used_for_fold_design"] is False
assert primary_validation["fold_boundaries_frozen_before_model_selection"] is True

NUMBER_OF_FOLDS = int(primary_validation["number_of_folds"])
FIRST_VALIDATION_START_FRACTION = float(
    primary_validation["first_validation_start_fraction"]
)
EMBARGO_TRADING_DAYS = int(
    primary_validation["embargo_trading_days"]
)

print("Candidate path:", CANDIDATE_TRAIN_PATH)
print("Calendar source:", LABELED_TRAIN_PATH)
print("Number of folds:", NUMBER_OF_FOLDS)
print(
    "First validation start fraction:",
    FIRST_VALIDATION_START_FRACTION,
)
print("Embargo trading days:", EMBARGO_TRADING_DAYS)
print("Unseen-test used for fold design: False")


Candidate path: E:\Iran_stock_trade\financial-ml-decision-support\data_ready\candidates\train
Calendar source: E:\Iran_stock_trade\financial-ml-decision-support\data_ready\labeled\train
Number of folds: 5
First validation start fraction: 0.5
Embargo trading days: 30
Unseen-test used for fold design: False


## 3. Validate frozen universe and Stage 04 candidate files

In [4]:
frozen_universe_df = pd.read_csv(
    frozen_universe_path,
    low_memory=False,
)
frozen_symbols = sorted(
    frozen_universe_df["symbol"].astype(str).tolist()
)
frozen_symbol_set = set(frozen_symbols)

candidate_files = sorted(
    CANDIDATE_TRAIN_PATH.glob("*_train_candidates.csv")
)
labeled_train_files = sorted(
    LABELED_TRAIN_PATH.glob("*_train_labeled.csv")
)


def candidate_symbol_from_path(path: Path) -> str:
    suffix = "_train_candidates.csv"
    if not path.name.endswith(suffix):
        raise ValueError(f"Unexpected candidate file: {path.name}")
    return path.name[:-len(suffix)]


def labeled_symbol_from_path(path: Path) -> str:
    suffix = "_train_labeled.csv"
    if not path.name.endswith(suffix):
        raise ValueError(f"Unexpected labeled file: {path.name}")
    return path.name[:-len(suffix)]


candidate_file_map = {
    candidate_symbol_from_path(path): path
    for path in candidate_files
}
labeled_file_map = {
    labeled_symbol_from_path(path): path
    for path in labeled_train_files
}

assert set(candidate_file_map) == frozen_symbol_set
assert set(labeled_file_map) == frozen_symbol_set
assert len(candidate_file_map) == len(frozen_symbols)
assert len(labeled_file_map) == len(frozen_symbols)

print("Frozen symbols:", len(frozen_symbols))
print("Candidate files:", len(candidate_file_map))
print("Labeled train files:", len(labeled_file_map))


Frozen symbols: 499
Candidate files: 499
Labeled train files: 499


## 4. Build the train-only trading calendar

The market calendar is built from frozen **labeled train** dates, not candidate
dates. This keeps the 30-day gap in trading-calendar observations.

Candidate-event counts are introduced only later to choose boundaries inside the
validation horizon. Candidate dates do not redefine the market calendar.


In [5]:
calendar_date_parts = []
calendar_error_rows = []
started = time.perf_counter()

for file_number, (symbol, path) in enumerate(
    sorted(labeled_file_map.items()),
    start=1,
):
    try:
        dates = pd.read_csv(
            path,
            usecols=["dEven"],
            low_memory=False,
        )["dEven"]
        calendar_date_parts.append(dates)
    except Exception as exc:
        calendar_error_rows.append(
            {
                "symbol": symbol,
                "file_name": path.name,
                "error_type": type(exc).__name__,
                "error_message": str(exc),
            }
        )

    if (
        file_number == 1
        or file_number % 50 == 0
        or file_number == len(labeled_file_map)
    ):
        elapsed = time.perf_counter() - started
        print(
            f"[calendar] "
            f"[{file_number:>4}/{len(labeled_file_map)}] "
            f"elapsed={elapsed:,.1f}s "
            f"errors={len(calendar_error_rows)}"
        )

calendar_errors_df = pd.DataFrame(
    calendar_error_rows,
    columns=[
        "symbol",
        "file_name",
        "error_type",
        "error_message",
    ],
)

all_calendar_dates = pd.concat(
    calendar_date_parts,
    ignore_index=True,
)
trading_calendar = normalize_trading_calendar(
    all_calendar_dates.tolist()
)

calendar_summary_df = pd.DataFrame(
    [
        {
            "calendar_source": "frozen_labeled_train",
            "unique_trading_dates": len(trading_calendar),
            "first_trading_date": trading_calendar.min(),
            "last_trading_date": trading_calendar.max(),
            "unseen_test_dates_used": False,
        }
    ]
)

calendar_errors_df.to_csv(
    RESULT_PATHS["audits"] / "05_calendar_errors.csv",
    index=False,
    encoding="utf-8-sig",
)
calendar_summary_df.to_csv(
    RESULT_PATHS["audits"] / "05_candidate_calendar_audit.csv",
    index=False,
    encoding="utf-8-sig",
)

display(calendar_summary_df)


[calendar] [   1/499] elapsed=0.0s errors=0
[calendar] [  50/499] elapsed=1.1s errors=0
[calendar] [ 100/499] elapsed=2.1s errors=0
[calendar] [ 150/499] elapsed=3.3s errors=0
[calendar] [ 200/499] elapsed=4.7s errors=0
[calendar] [ 250/499] elapsed=6.0s errors=0
[calendar] [ 300/499] elapsed=7.2s errors=0
[calendar] [ 350/499] elapsed=8.4s errors=0
[calendar] [ 400/499] elapsed=9.7s errors=0
[calendar] [ 450/499] elapsed=11.0s errors=0
[calendar] [ 499/499] elapsed=12.3s errors=0


,calendar_source,unique_trading_dates,first_trading_date,last_trading_date,unseen_test_dates_used
0,frozen_labeled_train,1690,2014-03-19,2021-03-17,False


## 5. Build the Stage 04 candidate-event panel

In [6]:
candidate_parts = []
candidate_error_rows = []
started = time.perf_counter()

usecols = [
    "dEven",
    "event_end_date",
    "meta_label",
]

for file_number, (symbol, path) in enumerate(
    sorted(candidate_file_map.items()),
    start=1,
):
    try:
        frame = pd.read_csv(
            path,
            usecols=usecols,
            low_memory=False,
        )
        frame.insert(0, "symbol", symbol)

        event_dates = pd.to_datetime(
            frame["dEven"],
            errors="coerce",
        )
        frame.insert(
            0,
            "event_id",
            (
                symbol
                + "::"
                + event_dates.dt.strftime("%Y-%m-%d")
            ),
        )
        candidate_parts.append(frame)

    except Exception as exc:
        candidate_error_rows.append(
            {
                "symbol": symbol,
                "file_name": path.name,
                "error_type": type(exc).__name__,
                "error_message": str(exc),
            }
        )

    if (
        file_number == 1
        or file_number % 50 == 0
        or file_number == len(candidate_file_map)
    ):
        elapsed = time.perf_counter() - started
        print(
            f"[candidate panel] "
            f"[{file_number:>4}/{len(candidate_file_map)}] "
            f"elapsed={elapsed:,.1f}s "
            f"errors={len(candidate_error_rows)}"
        )

candidate_errors_df = pd.DataFrame(
    candidate_error_rows,
    columns=[
        "symbol",
        "file_name",
        "error_type",
        "error_message",
    ],
)

candidate_panel = normalize_candidate_event_panel(
    pd.concat(candidate_parts, ignore_index=True)
)

candidate_panel_summary_df = pd.DataFrame(
    [
        {
            "candidate_events": len(candidate_panel),
            "candidate_symbols": candidate_panel["symbol"].nunique(),
            "first_event_start_date": candidate_panel["dEven"].min(),
            "last_event_start_date": candidate_panel["dEven"].max(),
            "positive_meta_labels": int(
                candidate_panel["meta_label"].eq(1).sum()
            ),
            "negative_meta_labels": int(
                candidate_panel["meta_label"].eq(0).sum()
            ),
            "positive_meta_label_fraction": float(
                candidate_panel["meta_label"].eq(1).mean()
            ),
            "duplicate_event_ids": int(
                candidate_panel["event_id"].duplicated().sum()
            ),
        }
    ]
)

candidate_errors_df.to_csv(
    RESULT_PATHS["audits"] / "05_candidate_panel_errors.csv",
    index=False,
    encoding="utf-8-sig",
)
candidate_panel_summary_df.to_csv(
    RESULT_PATHS["audits"] / "05_candidate_panel_audit.csv",
    index=False,
    encoding="utf-8-sig",
)

display(candidate_panel_summary_df)


candidate_event_counts_by_date = (
    candidate_event_counts_by_calendar_date(
        trading_calendar,
        candidate_panel["dEven"],
    )
)

candidate_calendar_audit_df = pd.DataFrame(
    {
        "dEven": candidate_event_counts_by_date.index,
        "candidate_event_count": (
            candidate_event_counts_by_date.to_numpy()
        ),
    }
)
candidate_calendar_audit_df["has_candidate_event"] = (
    candidate_calendar_audit_df["candidate_event_count"].gt(0)
)

candidate_calendar_audit_df.to_csv(
    RESULT_PATHS["audits"]
    / "05_candidate_calendar_audit.csv",
    index=False,
    encoding="utf-8-sig",
)

print(
    "Calendar dates with at least one candidate:",
    int(candidate_calendar_audit_df["has_candidate_event"].sum()),
)


[candidate panel] [   1/499] elapsed=0.1s errors=0
[candidate panel] [  50/499] elapsed=3.2s errors=0
[candidate panel] [ 100/499] elapsed=6.2s errors=0
[candidate panel] [ 150/499] elapsed=9.5s errors=0
[candidate panel] [ 200/499] elapsed=12.3s errors=0
[candidate panel] [ 250/499] elapsed=15.0s errors=0
[candidate panel] [ 300/499] elapsed=17.8s errors=0
[candidate panel] [ 350/499] elapsed=19.9s errors=0
[candidate panel] [ 400/499] elapsed=21.7s errors=0
[candidate panel] [ 450/499] elapsed=23.6s errors=0
[candidate panel] [ 499/499] elapsed=25.4s errors=0


,candidate_events,candidate_symbols,first_event_start_date,last_event_start_date,positive_meta_labels,negative_meta_labels,positive_meta_label_fraction,duplicate_event_ids
0,118464,499,2014-05-06,2021-03-14,63873,54591,0.539176,0


Calendar dates with at least one candidate: 1654


## 6. Freeze five event-count-balanced anchored folds

The first validation date is still determined by the frozen 50% calendar rule.

From that date onward, candidate-event start-date counts are accumulated over
the frozen trading calendar. Validation boundaries are selected near 20%, 40%,
60%, 80%, and 100% of the validation-horizon candidate count.

Because boundaries may occur only between whole calendar dates, the event counts
are approximately rather than exactly equal. `meta_label` is not supplied to the
fold-construction function.


In [7]:
folds = build_anchored_event_balanced_folds(
    trading_calendar,
    candidate_panel["dEven"],
    number_of_folds=NUMBER_OF_FOLDS,
    first_validation_start_fraction=(
        FIRST_VALIDATION_START_FRACTION
    ),
    embargo_trading_days=EMBARGO_TRADING_DAYS,
)

fold_boundaries_df = folds_to_frame(folds)

fold_boundaries_path = (
    RESULT_PATHS["folds"]
    / "05_purged_anchored_walk_forward_boundaries.csv"
)
fold_boundaries_df.to_csv(
    fold_boundaries_path,
    index=False,
    encoding="utf-8-sig",
)

display(fold_boundaries_df)


,fold_id,calendar_start_date,train_end_date,embargo_start_date,embargo_end_date,validation_start_date,validation_end_date,train_end_calendar_index,validation_start_calendar_index,validation_end_calendar_index,embargo_trading_days,validation_target_candidate_events,validation_candidate_events,validation_event_count_deviation,validation_event_count_relative_deviation,validation_horizon_candidate_events,validation_partition_method
0,1,2014-03-19,2017-08-06,2017-08-07,2017-09-18,2017-09-19,2018-01-31,814,845,936,30,10368.0,10344,-24.0,-0.002315,51840,candidate_event_count_balanced_contiguous_cale...
1,2,2014-03-19,2017-12-20,2017-12-23,2018-01-31,2018-02-03,2018-06-27,906,937,1027,30,10368.0,10408,40.0,0.003858,51840,candidate_event_count_balanced_contiguous_cale...
2,3,2014-03-19,2018-05-12,2018-05-13,2018-06-27,2018-06-30,2019-01-15,997,1028,1165,30,10368.0,10397,29.0,0.002797,51840,candidate_event_count_balanced_contiguous_cale...
3,4,2014-03-19,2018-12-04,2018-12-05,2019-01-15,2019-01-16,2020-01-04,1135,1166,1396,30,10368.0,10319,-49.0,-0.004726,51840,candidate_event_count_balanced_contiguous_cale...
4,5,2014-03-19,2019-11-23,2019-11-24,2020-01-04,2020-01-05,2021-03-17,1366,1397,1689,30,10368.0,10372,4.0,0.000386,51840,candidate_event_count_balanced_contiguous_cale...


## 7. Apply purge and embargo controls without fitting any model

For each fold:

1. the historical candidate population is restricted to starts on or before the
   pre-embargo training end;
2. any retained historical event with `event_end_date >= validation_start_date`
   is purged;
3. validation consists only of candidate events whose start dates fall inside
   that fold's contiguous validation window.


In [8]:
fold_summary_rows = []
fold_integrity_rows = []
validation_assignment_parts = []

for fold in folds:
    masks = fold_membership_masks(candidate_panel, fold)

    fold_summary_rows.append(
        summarize_fold(
            candidate_panel,
            fold,
            masks,
        )
    )
    fold_integrity_rows.append(
        audit_fold_integrity(
            candidate_panel,
            fold,
            masks,
        )
    )

    validation_events = candidate_panel.loc[
        masks.validation,
        [
            "event_id",
            "symbol",
            "dEven",
            "event_end_date",
            "meta_label",
        ],
    ].copy()
    validation_events.insert(0, "fold_id", fold.fold_id)
    validation_assignment_parts.append(validation_events)

fold_summary_df = pd.DataFrame(fold_summary_rows)
fold_integrity_df = pd.DataFrame(fold_integrity_rows)
validation_assignments_df = pd.concat(
    validation_assignment_parts,
    ignore_index=True,
)

anchored_audit_df = audit_anchored_training_sets(
    candidate_panel,
    folds,
)
validation_overlap_audit_df = audit_validation_window_overlap(
    folds
)
validation_event_count_balance_df = (
    audit_validation_event_count_balance(
        candidate_panel,
        folds,
    )
)

fold_summary_df.to_csv(
    RESULT_PATHS["folds"]
    / "05_purged_anchored_walk_forward_folds.csv",
    index=False,
    encoding="utf-8-sig",
)
validation_assignments_df.to_csv(
    RESULT_PATHS["folds"]
    / "05_validation_event_assignments.csv",
    index=False,
    encoding="utf-8-sig",
)
fold_integrity_df.to_csv(
    RESULT_PATHS["audits"] / "05_fold_integrity_audit.csv",
    index=False,
    encoding="utf-8-sig",
)
anchored_audit_df.to_csv(
    RESULT_PATHS["audits"] / "05_anchored_training_audit.csv",
    index=False,
    encoding="utf-8-sig",
)
validation_overlap_audit_df.to_csv(
    RESULT_PATHS["audits"]
    / "05_validation_window_overlap_audit.csv",
    index=False,
    encoding="utf-8-sig",
)
validation_event_count_balance_df.to_csv(
    RESULT_PATHS["audits"]
    / "05_validation_event_count_balance.csv",
    index=False,
    encoding="utf-8-sig",
)

display(fold_summary_df)
display(fold_integrity_df)
display(anchored_audit_df)


display(validation_event_count_balance_df)


,fold_id,train_start_date,train_end_date,embargo_start_date,embargo_end_date,validation_start_date,validation_end_date,historical_events_before_embargo,purged_event_overlap_count,embargo_candidate_event_count,...,train_negative_labels,train_positive_fraction,validation_events,validation_positive_labels,validation_negative_labels,validation_positive_fraction,train_symbols,validation_symbols,train_unique_event_start_dates,validation_unique_event_start_dates
0,1,2014-05-06,2017-08-06,2017-08-07,2017-09-18,2017-09-19,2018-01-31,63238,521,3386,...,31216,0.502272,10344,3990,6354,0.385731,443,395,786,92
1,2,2014-05-06,2017-12-20,2017-12-23,2018-01-31,2018-02-03,2018-06-27,73935,345,3033,...,37852,0.485637,10408,4857,5551,0.466660,469,408,878,91
2,3,2014-05-06,2018-05-12,2018-05-13,2018-06-27,2018-06-30,2019-01-15,84144,319,3232,...,44037,0.474656,10397,7552,2845,0.726363,486,461,969,136
3,4,2014-05-06,2018-12-04,2018-12-05,2019-01-15,2019-01-16,2020-01-04,94189,130,3584,...,47197,0.498219,10319,9164,1155,0.888071,495,469,1105,231
4,5,2014-05-06,2019-11-23,2019-11-24,2020-01-04,2020-01-05,2021-03-17,107372,12,720,...,49462,0.539288,10372,5260,5112,0.507135,499,453,1336,288


,fold_id,train_event_ids_in_validation,train_validation_event_start_date_overlap,train_start_after_train_end,train_event_end_on_or_after_validation_start,validation_before_validation_start,validation_after_validation_end,embargo_events_in_train,purged_events_in_train,train_has_both_classes,validation_has_both_classes
0,1,0,0,0,0,0,0,0,0,True,True
1,2,0,0,0,0,0,0,0,0,True,True
2,3,0,0,0,0,0,0,0,0,True,True
3,4,0,0,0,0,0,0,0,0,True,True
4,5,0,0,0,0,0,0,0,0,True,True


,previous_fold_id,current_fold_id,anchored_subset_ok,previous_train_events_missing_from_current,current_train_events
0,NaN,1,True,0,62717
1,1.0,2,True,0,73590
2,2.0,3,True,0,83825
3,3.0,4,True,0,94059
4,4.0,5,True,0,107360


,fold_id,validation_start_date,validation_end_date,target_candidate_events,actual_candidate_events,absolute_deviation_from_target,signed_deviation_from_target,relative_deviation_from_target,validation_unique_start_dates,boundary_used_meta_label,all_fold_event_count_mean,all_fold_event_count_std,all_fold_event_count_cv,all_fold_event_count_min,all_fold_event_count_max,all_fold_max_to_min_ratio,total_validation_candidate_events,common_target_candidate_events
0,1,2017-09-19,2018-01-31,10368.0,10344,24.0,-24.0,-0.002315,92,False,10368.0,32.96665,0.00318,10319,10408,1.008625,51840,10368.0
1,2,2018-02-03,2018-06-27,10368.0,10408,40.0,40.0,0.003858,91,False,10368.0,32.96665,0.00318,10319,10408,1.008625,51840,10368.0
2,3,2018-06-30,2019-01-15,10368.0,10397,29.0,29.0,0.002797,136,False,10368.0,32.96665,0.00318,10319,10408,1.008625,51840,10368.0
3,4,2019-01-16,2020-01-04,10368.0,10319,49.0,-49.0,-0.004726,231,False,10368.0,32.96665,0.00318,10319,10408,1.008625,51840,10368.0
4,5,2020-01-05,2021-03-17,10368.0,10372,4.0,4.0,0.000386,288,False,10368.0,32.96665,0.00318,10319,10408,1.008625,51840,10368.0


## 8. Descriptive class-balance audit after boundaries are frozen

`meta_label` was not used to select validation boundaries.

Only after the date boundaries are fixed do we report positive and negative
fractions. This preserves genuine market-regime shifts instead of stratifying
time folds by the observed outcome.


In [9]:
class_balance_rows = []

for row in fold_summary_df.itertuples(index=False):
    class_balance_rows.extend(
        [
            {
                "fold_id": row.fold_id,
                "partition": "train",
                "events": row.train_events,
                "positive_labels": row.train_positive_labels,
                "negative_labels": row.train_negative_labels,
                "positive_fraction": row.train_positive_fraction,
            },
            {
                "fold_id": row.fold_id,
                "partition": "validation",
                "events": row.validation_events,
                "positive_labels": row.validation_positive_labels,
                "negative_labels": row.validation_negative_labels,
                "positive_fraction": row.validation_positive_fraction,
            },
        ]
    )

fold_class_balance_df = pd.DataFrame(class_balance_rows)

fold_class_balance_df.to_csv(
    RESULT_PATHS["audits"] / "05_fold_class_balance.csv",
    index=False,
    encoding="utf-8-sig",
)

display(fold_class_balance_df)


,fold_id,partition,events,positive_labels,negative_labels,positive_fraction
0,1,train,62717,31501,31216,0.502272
1,1,validation,10344,3990,6354,0.385731
2,2,train,73590,35738,37852,0.485637
3,2,validation,10408,4857,5551,0.466660
4,3,train,83825,39788,44037,0.474656
5,3,validation,10397,7552,2845,0.726363
6,4,train,94059,46862,47197,0.498219
7,4,validation,10319,9164,1155,0.888071
8,5,train,107360,57898,49462,0.539288
9,5,validation,10372,5260,5112,0.507135


## 9. Freeze Stage 05 manifest

In [10]:
fold_records = json.loads(
    fold_boundaries_df.to_json(
        orient="records",
        date_format="iso",
    )
)

manifest = {
    "stage": 5,
    "status": "frozen_after_integrity_checks",
    "notebook": "05_purged_walk_forward_design.ipynb",
    "git_commit_sha": git_commit_sha(REPOSITORY_ROOT),
    "software": software_manifest(),
    "candidate_population": {
        "source": "Stage 04 primary long candidates",
        "events": len(candidate_panel),
        "symbols": int(candidate_panel["symbol"].nunique()),
        "primary_zigzag_threshold_fraction": 0.15,
        "target_column": "meta_label",
    },
    "calendar": {
        "source": "frozen_labeled_train",
        "unique_trading_dates": len(trading_calendar),
        "first_date": str(trading_calendar.min().date()),
        "last_date": str(trading_calendar.max().date()),
    },
    "primary_validation": primary_validation,
    "fold_design_schema_version": FOLD_DESIGN_SCHEMA_VERSION,
    "boundary_construction": {
        "input": "candidate_event_start_date_counts_only",
        "meta_label_used": False,
        "event_outcomes_used": False,
        "partition": (
            "approximately_equal_candidate_event_count_"
            "contiguous_calendar_blocks"
        ),
    },
    "validation_event_count_balance": json.loads(
        validation_event_count_balance_df.to_json(
            orient="records",
            date_format="iso",
        )
    ),
    "folds": fold_records,
    "unseen_test_used": False,
    "configuration_hash": stable_object_hash(
        {
            "validation": validation_config,
            "stage04_candidate_policy": stage04_policy,
        }
    ),
}

manifest_path = (
    RESULT_PATHS["manifests"]
    / "05_purged_anchored_walk_forward_manifest.json"
)
manifest_path.write_text(
    json.dumps(
        manifest,
        ensure_ascii=False,
        indent=2,
        default=str,
    ),
    encoding="utf-8",
)

print("Manifest:", manifest_path)


Manifest: E:\Iran_stock_trade\financial-ml-decision-support\results\manifests\05_purged_anchored_walk_forward_manifest.json


## 10. Final integrity checks

In [11]:
assert calendar_errors_df.empty, (
    "Calendar construction errors exist. "
    "Review 05_calendar_errors.csv"
)
assert candidate_errors_df.empty, (
    "Candidate panel errors exist. "
    "Review 05_candidate_panel_errors.csv"
)
assert len(folds) == NUMBER_OF_FOLDS
assert len(candidate_panel) > 0
assert candidate_panel["event_id"].is_unique
assert candidate_panel["meta_label"].isin([0, 1]).all()

assert fold_integrity_df[
    "train_event_ids_in_validation"
].eq(0).all()

assert fold_integrity_df[
    "train_validation_event_start_date_overlap"
].eq(0).all()

assert fold_integrity_df[
    "train_start_after_train_end"
].eq(0).all()

assert fold_integrity_df[
    "train_event_end_on_or_after_validation_start"
].eq(0).all()

assert fold_integrity_df[
    "validation_before_validation_start"
].eq(0).all()

assert fold_integrity_df[
    "validation_after_validation_end"
].eq(0).all()

assert fold_integrity_df[
    "embargo_events_in_train"
].eq(0).all()

assert fold_integrity_df[
    "purged_events_in_train"
].eq(0).all()

assert fold_integrity_df[
    "train_has_both_classes"
].all()

assert fold_integrity_df[
    "validation_has_both_classes"
].all()

assert anchored_audit_df[
    "anchored_subset_ok"
].all()

assert not validation_overlap_audit_df[
    "validation_windows_overlap"
].any()


assert fold_boundaries_df[
    "validation_candidate_events"
].gt(0).all()

assert fold_boundaries_df[
    "validation_partition_method"
].eq(
    "candidate_event_count_balanced_contiguous_calendar"
).all()

assert validation_event_count_balance_df[
    "boundary_used_meta_label"
].eq(False).all()

first_validation_start = min(
    fold.validation_start_date
    for fold in folds
)
expected_validation_horizon_events = int(
    candidate_panel["dEven"]
    .ge(first_validation_start)
    .sum()
)

assert int(
    validation_event_count_balance_df[
        "actual_candidate_events"
    ].sum()
) == expected_validation_horizon_events

assert len(
    validation_assignments_df
) == expected_validation_horizon_events

assert int(
    validation_event_count_balance_df[
        "total_validation_candidate_events"
    ].iloc[0]
) == expected_validation_horizon_events

assert validation_assignments_df["event_id"].is_unique
assert validation_assignments_df["fold_id"].between(
    1,
    NUMBER_OF_FOLDS,
).all()

print("Notebook 05 integrity checks: PASSED")
print("Candidate long events:", len(candidate_panel))
print("Walk-forward folds:", len(folds))
print(
    "Validation partition:",
    "candidate-event-count-balanced contiguous calendar blocks",
)
print("Label balance used for boundaries: False")
print(
    "Validation event counts:",
    validation_event_count_balance_df[
        "actual_candidate_events"
    ].astype(int).tolist(),
)
print(
    "Validation event-count CV:",
    float(
        validation_event_count_balance_df[
            "all_fold_event_count_cv"
        ].iloc[0]
    ),
)
print("Embargo gap:", EMBARGO_TRADING_DAYS, "trading days")
print("Purge method: event_end_overlap")
print(
    "Train/validation event overlap:",
    int(
        fold_integrity_df[
            "train_event_ids_in_validation"
        ].sum()
    ),
)
print(
    "Training label intervals reaching validation:",
    int(
        fold_integrity_df[
            "train_event_end_on_or_after_validation_start"
        ].sum()
    ),
)
print(
    "Validation window overlap pairs:",
    int(
        validation_overlap_audit_df[
            "validation_windows_overlap"
        ].sum()
    ),
)
print(
    "Anchored training subset failures:",
    int(
        (~anchored_audit_df["anchored_subset_ok"]).sum()
    ),
)
print("Unseen-test used in Stage 05 decisions: False")
print(
    "Next stage: Notebook 06 — Optuna model selection "
    "with frozen purged walk-forward folds."
)


Notebook 05 integrity checks: PASSED
Candidate long events: 118464
Walk-forward folds: 5
Validation partition: candidate-event-count-balanced contiguous calendar blocks
Label balance used for boundaries: False
Validation event counts: [10344, 10408, 10397, 10319, 10372]
Validation event-count CV: 0.0031796537244042494
Embargo gap: 30 trading days
Purge method: event_end_overlap
Train/validation event overlap: 0
Training label intervals reaching validation: 0
Validation window overlap pairs: 0
Anchored training subset failures: 0
Unseen-test used in Stage 05 decisions: False
Next stage: Notebook 06 — Optuna model selection with frozen purged walk-forward folds.


## Review before freezing Stage 05

Inspect:

- `results/folds/05_purged_anchored_walk_forward_boundaries.csv`
- `results/folds/05_purged_anchored_walk_forward_folds.csv`
- `results/folds/05_validation_event_assignments.csv`
- `results/audits/05_fold_integrity_audit.csv`
- `results/audits/05_anchored_training_audit.csv`
- `results/audits/05_validation_window_overlap_audit.csv`
- `results/audits/05_fold_class_balance.csv`
- `results/audits/05_validation_event_count_balance.csv`
- `results/audits/05_candidate_calendar_audit.csv`
- `results/manifests/05_purged_anchored_walk_forward_manifest.json`

Notebook 06 must consume these frozen fold boundaries. It must not redesign the
calendar split after seeing RF, XGBoost, Dummy, or Logistic Regression results.


The class-balance file is descriptive only. It must not be used to move the
frozen boundaries after seeing `meta_label`.
